# 8. Strong duality and Slater's condition  [CO 14.4]

TOML-MIRI — Maximum Likelihood Estimation and Convex Optimization, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

np.set_printoptions(precision=4, suppress=True)
from scipy.optimize import minimize, minimize_scalar

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 8.1 When Slater holds

**Task.** Take $\min\ (x_1-2)^2 + (x_2-1)^2$ s.t. $x_1^2 + x_2^2\le r^2$. Plot the feasible disc, the contours and the optimum. Compute $g(\lambda)$ in closed form, plot it, and compare $d^\star$ and $p^\star$ for $r = 0.5, 1, 3$.

With $a = (2, 1)$, $\|a\| = \sqrt5$: $L = \|x-a\|^2 + \lambda(\|x\|^2 - r^2)$, $\nabla_xL = 0\Rightarrow x(\lambda) = \frac{a}{1+\lambda}$, and

$$g(\lambda) = \|a\|^2\frac{\lambda}{1+\lambda} - \lambda r^2, \qquad \lambda^\star = \max\Big(0, \frac{\|a\|}{r} - 1\Big), \qquad p^\star = \big(\max(0, \|a\| - r)\big)^2.$$

### Prediction

- For $r = 0.5, 1$ the target $(2,1)$ is outside the disc: the optimum is the radial projection $r\,a/\|a\|$, with $\lambda^\star>0$. For $r = 3 > \sqrt5$ the target is inside: $\lambda^\star = 0$, $p^\star = 0$. In all cases $d^\star = p^\star$.

In [ ]:
a_t = np.array([2.0, 1.0]); na = np.linalg.norm(a_t)
g = lambda lam, r: na**2 * lam / (1 + lam) - lam * r**2
lam_star = lambda r: max(0.0, na / r - 1)
p_star = lambda r: max(0.0, na - r) ** 2

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for j, r in enumerate([0.5, 1.0, 3.0]):
    ax = axes[0, j]
    G1, G2 = np.meshgrid(np.linspace(-3.5, 3.5, 300), np.linspace(-3.5, 3.5, 300))
    ax.contour(G1, G2, (G1 - 2) ** 2 + (G2 - 1) ** 2, levels=12, colors="gray", linewidths=0.6)
    ax.add_patch(plt.Circle((0, 0), r, color="#cfe0f3"))
    x_opt = a_t / (1 + lam_star(r))
    ax.plot(*a_t, "ko", label="target (2, 1)"); ax.plot(*x_opt, "r*", ms=15, label=f"x* = {x_opt.round(3)}")
    ax.plot(0, 0, "g^", ms=10, label="strictly feasible x = 0")
    ax.set_aspect("equal"); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.legend(fontsize=8); ax.set_title(f"r = {r}")
    lam = np.linspace(0, 8, 400)
    axes[1, j].plot(lam, g(lam, r), label="g(λ)"); axes[1, j].axhline(p_star(r), color="r", ls="--", label=f"p* = {p_star(r):.4f}")
    axes[1, j].plot(lam_star(r), g(lam_star(r), r), "r*", ms=14, label=f"λ* = {lam_star(r):.3f}, d* = {g(lam_star(r), r):.4f}")
    axes[1, j].set_ylim(min(-2, g(8, r)), p_star(r) + 1); axes[1, j].legend(); axes[1, j].grid(alpha=0.3); axes[1, j].set_xlabel("λ")
plt.show()

### Verify numerically

In [ ]:
for r in [0.5, 1.0, 3.0]:
    num_p = minimize(lambda x: np.sum((x - a_t) ** 2), np.zeros(2), method="SLSQP",
                     constraints=[{"type": "ineq", "fun": lambda x: r**2 - x @ x}]).fun
    lam_grid = np.linspace(0, 20, 400_001); num_d = g(lam_grid, r).max()
    gn = np.array([minimize(lambda x: np.sum((x - a_t) ** 2) + l * (x @ x - r**2), np.zeros(2)).fun for l in [0.3, 1.0, 2.5]])
    check(f"r = {r}: closed-form g matches numerical inf_x L", np.allclose(gn, g(np.array([0.3, 1.0, 2.5]), r), atol=1e-6))
    check(f"r = {r}: p* = {num_p:.5f} = d* = {num_d:.5f}  (strong duality, Slater holds at x = 0)", np.isclose(num_p, num_d, atol=1e-6))

### $p^\star(r)$, $\lambda^\star(r)$ and the sensitivity $dp^\star/d(r^2)$

In [ ]:
rs = np.linspace(0.2, 3.5, 300)
ps = np.array([p_star(r) for r in rs]); ls = np.array([lam_star(r) for r in rs])
s = rs**2
dp_ds = np.gradient(ps, s)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.3))
ax1.plot(rs, ps, label="p*(r)"); ax1.plot(rs, ls, label="λ*(r)"); ax1.axvline(na, color="k", ls=":", label="r = ‖a‖ = √5")
ax1.legend(); ax1.grid(alpha=0.3); ax1.set_xlabel("r")
ax2.plot(rs, ls, lw=3, alpha=0.5, label="λ*(r)"); ax2.plot(rs, -dp_ds, "k--", label="−dp*/d(r²) (finite differences)")
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_xlabel("r")
plt.show()
check("λ*(r) = −dp*/d(r²)", np.allclose(ls[2:-2], -dp_ds[2:-2], atol=2e-2))

---
### Explain

**1. Exhibit a strictly feasible point.**

$x = 0$: $f_1(0) = 0 - r^2 < 0$ for every $r>0$. The constraint is convex (a disc) and the objective is convex, so Slater's condition holds and strong duality is guaranteed, as observed: $d^\star = p^\star$ for $r = 0.5$ ($p^\star = (\sqrt5 - 0.5)^2\approx3.014$), $r = 1$ ($\approx1.528$) and $r = 3$ ($0$).

**2. What happens to $\lambda^\star$ when $r$ is so large that the constraint is not active?**

For $r\ge\sqrt5$ the target $(2,1)$ lies inside the disc, the unconstrained minimiser is feasible, the constraint is inactive, and $\lambda^\star = 0$ (complementary slackness; $g$ is then decreasing on $\lambda\ge0$ and peaks at 0). For $r<\sqrt5$, $\lambda^\star = \sqrt5/r - 1 > 0$, growing without bound as $r\to0$: squeezing the disc makes the constraint more and more "expensive".

**3. Plot $p^\star(r)$ and $\lambda^\star(r)$: how is $\lambda^\star$ related to $dp^\star/d(r^2)$?**

$\lambda^\star = -\frac{dp^\star}{d(r^2)}$: the multiplier is the **sensitivity of the optimal value to the right-hand side** of the constraint $\|x\|^2\le r^2$ (the constraint level is $r^2$, not $r$). Check: $p^\star = (\sqrt5 - r)^2$ gives $\frac{dp^\star}{dr} = -2(\sqrt5 - r)$ and $\frac{dp^\star}{d(r^2)} = \frac{1}{2r}\frac{dp^\star}{dr} = -(\frac{\sqrt5}{r} - 1) = -\lambda^\star$, confirmed by finite differences. It is the same "shadow price" interpretation as for the LP in 5.2: relaxing the constraint by one unit decreases the minimum by $\lambda^\star$ units. For $r>\sqrt5$ both are 0: extra room is worthless once the constraint is not binding.

---
## 8.2 When Slater fails

**Task.** (a) Let $r\to0$ above, then study $\min x$ s.t. $x^2\le0$: compute $g(\lambda) = -1/(4\lambda)$, plot it and find $d^\star$. (b) Study the classical example

$$\min_{x,\ y>0}\ e^{-x}\quad\text{s.t.}\quad x^2/y\le0.$$

Plot the feasible set, compute $p^\star$ and show numerically that $g(\lambda) = 0$ for all $\lambda\ge0$ (minimise $L$ along curves $x = t$, $y = t^3$, $t\to\infty$).

### Prediction

- (a) The feasible set is the single point $\{0\}$, $p^\star = 0$; $g(\lambda) = -1/(4\lambda)\to0$ but never reaches it: $d^\star = 0$ (no gap) but the dual optimum is **not attained**.
- (b) Feasible set: $x = 0$, $y>0$, so $p^\star = e^0 = 1$. The Lagrangian can be pushed to 0 by going far out with $y\gg x^2$: $d^\star = 0$, a gap of 1.

In [ ]:
# (a) min x  s.t. x² ≤ 0
lam = np.linspace(0.01, 50, 2000)
g_a = -1 / (4 * lam)
g_num = np.array([minimize_scalar(lambda x: x + l * x**2).fun for l in lam[::100]])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
ax1.plot(lam, g_a, label="g(λ) = −1/(4λ)"); ax1.plot(lam[::100], g_num, "o", ms=4, label="numerical")
ax1.axhline(0, color="r", ls="--", label="p* = 0 (sup g, never reached)"); ax1.set_ylim(-2, 0.3); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title("(a) min x s.t. x² ≤ 0"); ax1.set_xlabel("λ")

# (b) feasible set and the Lagrangian along x = t, y = t³
Xg, Yg = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(0.01, 3, 300))
ax2.contourf(Xg, Yg, np.exp(-Xg), levels=20, cmap="viridis", alpha=0.6)
ax2.plot([0, 0], [0.01, 3], "r", lw=4, label="feasible set {x = 0, y > 0}")
tt = np.linspace(0.5, 1.44, 50); ax2.plot(tt, tt**3, "w--", label="curve x = t, y = t³")
ax2.set_xlabel("x"); ax2.set_ylabel("y"); ax2.legend(); ax2.set_title("(b) e^{−x} (colour) and the feasible set")
plt.show()

ts = np.logspace(0, 4, 9)
print("  t        L(t, t³, λ) for λ = 0.1, 1, 10, 100")
for t in ts:
    print(f"{t:8.0f}   " + "   ".join(f"{np.exp(-t) + l * t**2 / t**3:.2e}" for l in [0.1, 1, 10, 100]))

### Verify numerically

In [ ]:
check("(a) closed form g(λ) = −1/(4λ) matches the numerical minimum", np.allclose(g_num, g_a[::100], atol=1e-8))
check("(a) g(λ) < 0 = p* for every λ > 0, and sup g = 0 (approached, not attained)", np.all(g_a < 0) and g_a.max() > -0.01)
L_b = lambda t, l: np.exp(-t) + l / t
check("(b) L along x = t, y = t³ tends to 0 for every λ (t = 10⁴)", all(L_b(1e4, l) < 2e-2 for l in [0.1, 1, 10, 100]))
rng = np.random.default_rng(0)
xr, yr, lr = rng.uniform(-10, 10, 100_000), rng.uniform(1e-3, 10, 100_000), rng.uniform(0, 100, 100_000)
check("(b) L(x, y, λ) ≥ 0 on 100,000 random points of the domain, so g(λ) = inf L = 0", np.all(np.exp(-xr) + lr * xr**2 / yr >= 0))
Xf, Yf = np.meshgrid(np.linspace(-3, 3, 301), np.linspace(0.01, 3, 300))
feas = Xf**2 / Yf <= 0
p_b = np.exp(-Xf[feas]).min()
check(f"(b) grid: feasible points are exactly x = 0, p* = {p_b:.3f}; with d* = 0 the duality gap is 1",
      np.all(Xf[feas] == 0) and np.isclose(p_b, 1))
check("(b) no point satisfies x²/y < 0 (Slater fails)", np.all(Xg**2 / Yg >= 0))

---
### Explain

**1. In (a), is there a duality gap? Is the dual optimum attained?**

**No gap, but the dual optimum is not attained.** $L = x + \lambda x^2$ is minimised at $x = -\frac{1}{2\lambda}$, giving $g(\lambda) = -\frac{1}{4\lambda}$, which increases towards 0 as $\lambda\to\infty$ but is negative for every finite $\lambda$. So $d^\star = \sup_\lambda g = 0 = p^\star$, yet no $\lambda^\star$ achieves it. Slater fails (no $x$ with $x^2<0$; the feasible set $\{0\}$ has no interior), and with it the KKT conditions: stationarity $1 + 2\lambda x^\star = 1 \ne 0$ at $x^\star = 0$ for every $\lambda$. This is the $r\to0$ limit of 8.1, where $\lambda^\star = \sqrt5/r - 1\to\infty$: the price of the constraint becomes infinite.

**2. In (b), what is the duality gap and why can Slater's condition not hold?**

The domain is $y>0$, so $x^2/y\le0$ forces $x = 0$: the feasible set is the half-line $\{(0,y) : y>0\}$ and $p^\star = e^0 = 1$. For any $\lambda\ge0$, $L = e^{-x} + \lambda x^2/y\ge0$, and along $x = t$, $y = t^3$ it equals $e^{-t} + \lambda/t\to0$ (the table shows it). So $g(\lambda) = 0$ for all $\lambda$, $d^\star = 0$, and the **duality gap is 1**. Slater needs a point with $x^2/y<0$, which is impossible since $x^2/y\ge0$ on the domain. The feasible set lies entirely on the boundary $f_1 = 0$ of the constraint, so there is no strictly feasible point. (The constraint function $x^2/y$ is convex on $y>0$, a quadratic-over-linear function, so the problem *is* convex.)

**3. Both problems are convex: what does this tell you about convexity alone?**

**Convexity alone does not guarantee strong duality** (nor that the dual optimum is attained, nor that KKT points exist). It guarantees that the dual is a concave maximisation and that local minima are global, but zero gap needs an extra **constraint qualification** such as Slater's condition (a strictly feasible point). In (a) the gap is zero only by luck, and the dual is not attained. In (b) there is a genuine gap of 1. In practice most convex problems satisfy Slater (any problem with a strictly feasible point and affine equalities), which is why strong duality "usually" holds for convex problems [CO 14.4]; these two examples are the pathological boundary cases where the feasible set is "thin".